## Deploy NVIDIA-hosted NIMs

Run this notebook to deploy the Docker Compose version of the AI Virtual Assistant that uses NVIDIA-hosted NIMs and CPU Milvus. Paste your NVIDIA API key into the first cell's input box, then run each cell in order.

For the simplest one-key flow, use an NGC personal key that includes both `NGC Catalog` and `Cloud Functions` services. The notebook will use that same key for hosted inference and Docker image pulls. If you have separate keys, paste the Docker/NGC key into the optional second input box.

In [ ]:
import getpass
import os

NVIDIA_API_KEY = os.environ.get("NVIDIA_API_KEY", "")
if not NVIDIA_API_KEY:
    NVIDIA_API_KEY = getpass.getpass("Enter your NVIDIA API key: ")

if not NVIDIA_API_KEY:
    raise ValueError("NVIDIA_API_KEY is required.")

NGC_API_KEY = os.environ.get("NGC_API_KEY", "")
if not NGC_API_KEY:
    NGC_API_KEY = getpass.getpass("Optional Docker/NGC API key (press Enter to reuse NVIDIA_API_KEY): ") or NVIDIA_API_KEY

if not NGC_API_KEY:
    raise ValueError("NGC_API_KEY is required for Docker image pulls.")

os.environ["NVIDIA_API_KEY"] = NVIDIA_API_KEY
os.environ["NGC_API_KEY"] = NGC_API_KEY

print("Keys loaded. NVIDIA_API_KEY is set. NGC_API_KEY is set.")

In [ ]:
from pathlib import Path
import os

def find_repo_root():
    start = Path.cwd().resolve()
    for path in (start, *start.parents):
        if (path / "deploy" / "compose" / "docker-compose.yaml").exists():
            return path

    for candidate in [Path.home() / "ai-virtual-assistant", Path("/home/ubuntu/ai-virtual-assistant")]:
        if (candidate / "deploy" / "compose" / "docker-compose.yaml").exists():
            return candidate

    raise FileNotFoundError("Could not find deploy/compose/docker-compose.yaml. Open this notebook from the ai-virtual-assistant repo.")

REPO_ROOT = find_repo_root()
COMPOSE_FILE = REPO_ROOT / "deploy" / "compose" / "docker-compose.yaml"
ENV_FILE = REPO_ROOT / ".env.launchable"
os.chdir(REPO_ROOT)

print(f"Repository root: {REPO_ROOT}")
print(f"Compose file: {COMPOSE_FILE}")

In [ ]:
ENV_FILE.write_text(
    "\n".join([
        f"NVIDIA_API_KEY={NVIDIA_API_KEY}",
        f"NGC_API_KEY={NGC_API_KEY}",
        "APP_LLM_MODELNAME=nvidia/nemotron-3-nano-30b-a3b",
        "APP_VECTORSTORE_INDEXTYPE=IVF_FLAT",
        "",
    ])
)
ENV_FILE.chmod(0o600)

print(f"Wrote environment file: {ENV_FILE}")

In [ ]:
import subprocess
from collections import deque

LOG_DIR = REPO_ROOT / "logs"
LOG_DIR.mkdir(exist_ok=True)
DEPLOY_LOG = LOG_DIR / "deploy_hosted_nims.log"

def docker_cmd(*args):
    probe = subprocess.run(["docker", "info"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    if probe.returncode == 0:
        return ["docker", *args]
    return ["sudo", "docker", *args]

def run_logged(command, log_file, error_message):
    recent_lines = deque(maxlen=8)

    with log_file.open("a", encoding="utf-8") as log:
        log.write(f"\n\n$ {' '.join(str(part) for part in command)}\n")
        process = subprocess.Popen(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )

        for line in process.stdout:
            log.write(line)
            log.flush()
            stripped = line.strip()
            if stripped:
                recent_lines.append(stripped)

        return_code = process.wait()

    if return_code != 0:
        print(error_message)
        print(f"Full Docker output: {log_file}")
        print("Last Docker output lines:")
        for line in recent_lines:
            print(line)
        raise RuntimeError(f"Command failed with exit code {return_code}.")

    return return_code

version = subprocess.run(docker_cmd("compose", "version"), text=True, capture_output=True, check=True)
print(version.stdout.strip())

login = subprocess.run(
    docker_cmd("login", "nvcr.io", "-u", "$oauthtoken", "--password-stdin"),
    input=NGC_API_KEY,
    text=True,
    capture_output=True,
)
if login.returncode != 0:
    print(login.stdout)
    print(login.stderr)
    raise RuntimeError("Docker login to nvcr.io failed. If you used a Build API key, use an NGC personal key for NGC_API_KEY.")

print("Docker is ready and authenticated with nvcr.io.")

In [ ]:
config = subprocess.run(
    docker_cmd("compose", "--env-file", str(ENV_FILE), "-f", str(COMPOSE_FILE), "config"),
    text=True,
    capture_output=True,
)
if config.returncode != 0:
    print(config.stdout)
    print(config.stderr)
    raise RuntimeError("Docker Compose config validation failed.")

rendered = config.stdout
assert "nvidia/nemotron-3-nano-30b-a3b" in rendered
assert "milvusdb/milvus:v2.4.15" in rendered
assert "v2.4.15-gpu" not in rendered
assert "KNOWHERE_GPU_MEM_POOL_SIZE" not in rendered

print("Docker Compose configuration validated for hosted Nemotron 3 Nano and CPU Milvus.")

In [ ]:
DEPLOY_LOG.write_text("", encoding="utf-8")

run_logged(
    docker_cmd("compose", "--env-file", str(ENV_FILE), "-f", str(COMPOSE_FILE), "up", "-d", "--build"),
    DEPLOY_LOG,
    "Docker Compose deployment failed.",
)

print("Deployment started. Full Docker output was saved to logs/deploy_hosted_nims.log.")

In [ ]:
ps = subprocess.run(
    docker_cmd("compose", "--env-file", str(ENV_FILE), "-f", str(COMPOSE_FILE), "ps"),
    text=True,
    capture_output=True,
)
print(ps.stdout)
if ps.returncode != 0:
    print(ps.stderr)

print("\nOpen the sample UI on port 3001 once services are healthy.")

## Prepare data for ingestion

Download the sample product manuals so `notebooks/ingest_data.ipynb` can ingest PDFs without a missing `data/manuals_pdf` directory.

In [ ]:
from urllib.parse import unquote, urlparse
from urllib.request import urlretrieve

manual_list = REPO_ROOT / "data" / "list_manuals.txt"
manual_dir = REPO_ROOT / "data" / "manuals_pdf"
manual_dir.mkdir(parents=True, exist_ok=True)

downloaded = []
skipped = []

for url in manual_list.read_text(encoding="utf-8").splitlines():
    url = url.strip()
    if not url or url.startswith("#"):
        continue

    filename = Path(unquote(urlparse(url).path)).name
    target = manual_dir / filename
    if target.exists() and target.stat().st_size > 0:
        skipped.append(filename)
        continue

    print(f"Downloading {filename}...")
    urlretrieve(url, target)
    downloaded.append(filename)

print(f"Manuals ready in {manual_dir}")
print(f"Downloaded {len(downloaded)} file(s); skipped {len(skipped)} existing file(s).")
print("Next: open notebooks/ingest_data.ipynb to load the sample structured and unstructured data.")